# CogniNPC — Test 1: Benchmark wydajnosci i opoznien

 Wersja przeznaczona do generowania wynikow koncowych do pracy magisterskiej.

 **Zmienne niezalezne:** rozmiar modelu, wielkosc kontekstu RAG (k), wspolbieznosc (N)

 **Zmienne zalezne:** TTFT (stan ustalony i zimny start), TPS, czas calkowity,
 narzut orkiestracji, szczytowe zuzycie VRAM

 **Protokol:** rozgrzewka per warunek -> 30 powtorzen -> zapis surowych danych
 do JSONL -> analiza offline. Rozdzielenie zbierania od analizy pozwala
 przeliczac statystyki bez powtarzania kilkugodzinnego przebiegu.


In [ ]:
import json
import platform
import statistics
import subprocess
import threading
import time
import uuid
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from itertools import product
from pathlib import Path

import httpx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats  

sns.set_theme(style="whitegrid", context="paper", font_scale=1.05)
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"

OUT = Path("results")
(OUT / "figures").mkdir(parents=True, exist_ok=True)
(OUT / "tables").mkdir(parents=True, exist_ok=True)

API_URL = "http://localhost:8000/api/chat"
OLLAMA_URL = "http://localhost:11434"

NPC_IDS = ["thorin_01", "elara_01"]

SEND_MODEL_PARAM = True
SEND_RAG_K_PARAM = True

MODEL_NAME = "llama3.1:latest"

RAG_KS = [0, 3, 5, 10, 20] if SEND_RAG_K_PARAM else [None]
CONCURRENCIES = [1, 2, 4]
NUM_PREDICT = 128

# --- Protokol pomiarowy ----------------------------------------------------
REPETITIONS = 2                 # minimum 30 dla sensownego p95 i przedzialu ufnosci
WARMUP = 3                      # odrzucane z analizy: eliminuja zimny start
COOLDOWN_S = 0.4                # ogranicza kumulacje ciepla i throttling GPU
TIMEOUT_S = 300.0
MAX_RETRIES = 2

# Prog detekcji przeladowania wag: WZGLEDNY, nie absolutny.
RELOAD_RATIO_THRESHOLD = 3.0
RELOAD_FLOOR_MS = 50.0

# --- Progi interpretacyjne ----------------------------------
# Nielsen (1993) podaje trzy granice: 0,1 s, 1 s oraz 10 s. Wartosc 2000 ms
# NIE pochodzi od Nielsena — jest autorska definicja operacyjna budzetu
# maskowania animacja "NPC formuluje odpowiedz". W tekscie pracy nalezy
# te dwie rzeczy rozdzielic, aby nie przypisywac Nielsenowi progu,
# ktorego nie sformulowal.
T_FLOW_MS = 1000.0      # Nielsen 1993 - granica plynnosci toku myslenia
T_MASKED_MS = 2000.0    # definicja wlasna - akceptowalne przy maskowaniu animacja "NPC mysli"
T_BREAK_MS = 10000.0    # Nielsen 1993 - utrata uwagi uzytkownika

# --- Zestaw promptow --------------------------------------------------------
# Klasa promptu jest wspolzmienna kontrolowana: rotujemy je rownomiernie,
# zeby wynik nie zalezal od jednego szczegolnie latwego zapytania.
TEST_PROMPTS = [
    ("krotki", "Hello."),
    ("tozsamosc", "Who are you and what do you do here?"),
    ("prosba", "Can you forge a magical sword for me?"),
    ("emocjonalny", "What is your opinion on magic and elves? I heard you hate them."),
    ("zlozony", "I need a complete set of steel armor, a shield, and a heavy warhammer. "
                "How much will it cost and how long will it take?"),
]

BACKEND_ERROR_PREFIX = "System error:"

RUN_ID = uuid.uuid4().hex[:12]
#RUN_ID = "08df3ac72878"
RAW_PATH = OUT / f"raw_{RUN_ID}.jsonl"
print(f"run_id = {RUN_ID}\nmodel  = {MODEL_NAME}\nsurowe dane -> {RAW_PATH}")


 ## 2. Wymagane pola po stronie backendu

 `/api/chat` musi przyjmowac `model`, `rag_k`, `num_predict` oraz
 `skip_memory_write` (patrz patch_schemas.py / patch_npc_service.py /
 patch_router.py / patch_ollama_service.py), a `metrics` w odpowiedzi musi
 zawierac WSZYSTKIE ponizsze klucze — brak ktoregokolwiek cicho zeruje
 odpowiadajaca mu kolumne w analizie zamiast rzucic bledem:

  - `wall_total_ms`           -> czas scienny metody process_chat
  - `orchestration_ms`        -> wielkosc resztowa wg definicji z 5.2.4
  - `stage_rag_ms`            -> etap wyszukiwania wektorowego
  - `stage_prompt_build_ms`   -> etap budowy promptu

  Notebook NIE przelicza juz orchestration_ms samodzielnie — poprzednia
  wersja nadpisywala wartosc backendu inna definicja (wzgledem
  total_duration zamiast sumy trzech skladowych), co rozjezdzalo sie
  z opisem w pracy.

 response_chars w analizie czyta klucz `response_text` z ChatResponse —
 jesli nazwa pola w Twoim schemacie jest inna, zmien w single_request().

## 3. Telemetria VRAM i metadane srodowiska

In [ ]:
def nvidia_query(field: str) -> str | None:
    try:
        out = subprocess.check_output(
            ["nvidia-smi", f"--query-gpu={field}", "--format=csv,noheader,nounits"],
            stderr=subprocess.DEVNULL, timeout=5,
        )
        return out.decode().strip().splitlines()[0].strip()
    except Exception:
        return None


class VramSampler:
    """Probkuje zajetosc VRAM w watku tla; raportuje wartosc szczytowa."""

    def __init__(self, interval_s: float = 0.25):
        self.interval_s = interval_s
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None
        self._samples: list[float] = []
        self.available = nvidia_query("memory.used") is not None

    def _loop(self):
        while not self._stop.is_set():
            v = nvidia_query("memory.used")
            if v:
                try:
                    self._samples.append(float(v))
                except ValueError:
                    pass
            self._stop.wait(self.interval_s)

    def start(self):
        if not self.available:
            return
        self._samples = []
        self._stop.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def stop(self) -> dict:
        if not self.available:
            return {}
        self._stop.set()
        if self._thread:
            self._thread.join(timeout=2.0)
        if not self._samples:
            return {}
        return {
            "vram_peak_mb": round(max(self._samples), 1),
            "vram_mean_mb": round(statistics.fmean(self._samples), 1),
        }


def env_fingerprint() -> dict:
    def sh(cmd):
        try:
            return subprocess.check_output(cmd, stderr=subprocess.DEVNULL, timeout=10).decode().strip()
        except Exception:
            return None

    return {
        "git_commit": sh(["git", "rev-parse", "--short", "HEAD"]),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "gpu_name": nvidia_query("name"),
        "gpu_total_vram_mb": nvidia_query("memory.total"),
        "driver": nvidia_query("driver_version"),
    }


ENV = env_fingerprint()
print(json.dumps(ENV, indent=2, ensure_ascii=False))

# nvidia-smi raportuje zajetosc CALEJ karty, nie procesu. Bez wartosci
# odniesienia liczba w tabeli opisuje stan systemu, a nie koszt systemu
# CogniNPC. Roznica wzgledem baseline jest wielkoscia interpretowalna.
_v0 = nvidia_query("memory.used")
VRAM_BASELINE_MB = float(_v0) if _v0 else None
print(f"\nVRAM zajete przed przebiegiem (baseline): {VRAM_BASELINE_MB} MB")
print("UWAGA: uruchom te komorke PRZED zaladowaniem modelu do VRAM,")
print("       inaczej baseline bedzie juz zawieral wagi modelu.")

## 4. Pojedyncze wywolanie i dekompozycja metryk

In [ ]:
NS_TO_MS = 1e6

def parse_metrics(m: dict, baseline_load_ms: float | None = None) -> dict:
    """Przelicza natywne liczniki Ollamy (nanosekundy) na milisekundy.

    Rozdzielenie TTFT na wariant zimny i cieply jest istotne metodologicznie:
    `load_duration` jest niezerowe TYLKO gdy nastapilo ladowanie wag do VRAM.
    Wliczanie go do kazdego pomiaru mieszalo by dwa rozne zjawiska —
    jednorazowy koszt startu gry i powtarzalny koszt kazdej wypowiedzi.
    """
    load_ms = m.get("load_duration", 0) / NS_TO_MS
    prefill_ms = m.get("prompt_eval_duration", 0) / NS_TO_MS
    decode_ms = m.get("eval_duration", 0) / NS_TO_MS
    ollama_total_ms = m.get("total_duration", 0) / NS_TO_MS

    prompt_tokens = m.get("prompt_eval_count", 0)
    output_tokens = m.get("eval_count", 0)

    if baseline_load_ms and baseline_load_ms > 0:
        threshold = max(RELOAD_RATIO_THRESHOLD * baseline_load_ms, RELOAD_FLOOR_MS)
    else:
        threshold = RELOAD_FLOOR_MS
    reloaded = load_ms > threshold

    return {
        "llm_error": m.get("error"),
        "load_ms": round(load_ms, 2),
        "model_reloaded": reloaded,
        # Stan ustalony — to odczuwa gracz podczas rozgrywki:
        "ttft_warm_ms": round(prefill_ms, 2),
        # Pierwsza interakcja po uruchomieniu gry:
        "ttft_cold_ms": round(load_ms + prefill_ms, 2),
        "decode_ms": round(decode_ms, 2),
        "ollama_total_ms": round(ollama_total_ms, 2),
        "prompt_tokens": prompt_tokens,
        "output_tokens": output_tokens,
        # Normalizacja dlugosci wypowiedzi. Bez niej porownanie 8B vs 3.8B
        # jest bezwartosciowe: mniejszy model moze po prostu generowac dluzej.
        "tps": round(output_tokens / (decode_ms / 1000), 2) if decode_ms > 0 else None,
        "prefill_tps": round(prompt_tokens / (prefill_ms / 1000), 2) if prefill_ms > 0 else None,

        # Metryki zwrócone przez CogniNPC
        "orchestration_ms": m.get("orchestration_ms"),
        "wall_total_ms":    m.get("wall_total_ms"),
        "stage_rag_ms":     m.get("stage_rag_ms"),
        "stage_prompt_build_ms": m.get("stage_prompt_build_ms"),
        "rag_k_requested": m.get("rag_k_requested"),
        "rag_k_returned": m.get("rag_k_returned"),
    }


def single_request(client: httpx.Client, npc_id: str, prompt: str,
                   model: str, rag_k, prompt_class: str,
                   baseline_load_ms: float | None = None) -> dict:
    payload = {
        "npc_id": npc_id,
        "player_message": prompt,
        "skip_memory_write": True,
        "num_predict": NUM_PREDICT
        }
    if SEND_MODEL_PARAM:
        payload["model"] = model
    if SEND_RAG_K_PARAM and rag_k is not None:
        payload["rag_k"] = rag_k

    last_err = None
    for attempt in range(MAX_RETRIES + 1):
        # perf_counter, nie time.time() — tylko ten pierwszy jest monotoniczny
        # i odporny na korekty zegara systemowego w trakcie dlugiego przebiegu.
        t0 = time.perf_counter()
        try:
            r = client.post(API_URL, json=payload)
            wall_ms = (time.perf_counter() - t0) * 1000
            r.raise_for_status()
            body = r.json()
            rec = {"ok": True, "attempt": attempt, "wall_ms": round(wall_ms, 2)}
            rec.update(parse_metrics(body.get("metrics", {}), baseline_load_ms))

            resp_text = body.get("response_text", "") or ""
            rec["response_chars"] = len(resp_text)

            rec["response_preview"] = resp_text[:150]
            rec["prompt_class"] = prompt_class

            rec["backend_fallback"] = resp_text.startswith(BACKEND_ERROR_PREFIX)

            # Koszt walidacji FastAPI i transportu HTTP, nie modelu. To co jest wykluczone z definicji orchestration_ms w CogniNPC
            if rec.get("wall_total_ms"):
                rec["transport_ms"] = round(wall_ms - rec["wall_total_ms"], 2)
            else:
                rec["transport_ms"] = None
            return rec
        except Exception as exc:
            last_err = f"{type(exc).__name__}: {exc}"
            time.sleep(1.0)

    return {"ok": False, "error": last_err, "prompt_class": prompt_class,
            "attempt": MAX_RETRIES}

## 5. Wykonanie serii wspolbieznej

Ollama serializuje czesc pracy na jednym GPU, wiec latencja przy N=1 nie
mowi nic o scenie, w ktorej rozmawia kilku NPC jednoczesnie. Zakres
ograniczono do N<=4, co odpowiada realistycznej gestosci rownoczesnych
interakcji w scenie RPG przy pojedynczej karcie graficznej.

In [ ]:
def run_batch(client: httpx.Client, model: str, rag_k, concurrency: int, rep: int,
              baseline_load_ms: float | None = None) -> list[dict]:
    jobs = []
    for slot in range(concurrency):
        npc_id = NPC_IDS[slot % len(NPC_IDS)]
        cls, prompt = TEST_PROMPTS[(rep + slot) % len(TEST_PROMPTS)]
        jobs.append((npc_id, prompt, cls))

    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        futures = [
            ex.submit(single_request, client, npc, pr, model, rag_k, cls, baseline_load_ms)
            for npc, pr, cls in jobs
        ]
        results = [f.result() for f in futures]
    batch_ms = (time.perf_counter() - t0) * 1000

    for slot, rec in enumerate(results):
        rec["slot"] = slot
        # Czas domkniecia calej sceny — dla gracza istotniejszy niz czas
        # pojedynczej odpowiedzi, gdy w tle rozmawia kilku NPC.
        rec["batch_wall_ms"] = round(batch_ms, 2)
    return results

## 6. Glowna petla eksperymentu

Kolejnosc iteracji ustawiona tak, by model zmienial sie najwolniej —
minimalizuje to liczbe przeladowan wag do VRAM.

In [ ]:
conditions = [
    {"model": MODEL_NAME, "rag_k": k, "concurrency": c}
    for c, k in product(CONCURRENCIES, RAG_KS)
]
n_requests = sum(c["concurrency"] * (REPETITIONS + WARMUP) for c in conditions)
print(f"Warunkow: {len(conditions)} | Zadan lacznie: ~{n_requests}")

sampler = VramSampler()
if not sampler.available:
    print("UWAGA: nvidia-smi niedostepne — pomiar VRAM zostanie pominiety.")

last_model = None
t_start = time.perf_counter()

with RAW_PATH.open("w", encoding="utf-8") as fh, \
     httpx.Client(timeout=TIMEOUT_S, limits=httpx.Limits(max_connections=32)) as client:

    for ci, cond in enumerate(conditions, 1):
        model, rag_k, conc = cond["model"], cond["rag_k"], cond["concurrency"]

        label = f"{model} | k={rag_k} | N={conc}"
        print(f"[{ci}/{len(conditions)}] {label}", flush=True)

        # Rozgrzewka: absorbuje ladowanie wag i inicjalizacje polaczen HTTP.
        warmup_load_samples = []
        for w in range(WARMUP):
            recs = run_batch(client, model, rag_k, conc, rep=w)
            warmup_load_samples += [r["load_ms"] for r in recs if r.get("ok") and "load_ms" in r]
            time.sleep(COOLDOWN_S)

        baseline_load_ms = (
            statistics.median(warmup_load_samples) if warmup_load_samples else None
        )
        if baseline_load_ms:
            print(f"      baseline load_ms (z rozgrzewki): {baseline_load_ms:.1f}")

        sampler.start()
        for rep in range(REPETITIONS):
            recs = run_batch(client, model, rag_k, conc, rep, baseline_load_ms)
            for rec in recs:
                fh.write(json.dumps({
                    "run_id": RUN_ID,
                    "ts": datetime.now(timezone.utc).isoformat(),
                    "rep": rep,
                    **cond,
                    **rec,
                    **{f"env_{k}": v for k, v in ENV.items()},
                }, ensure_ascii=False) + "\n")
            fh.flush()
            time.sleep(COOLDOWN_S)
        vram = sampler.stop()

        # VRAM przypisujemy do warunku, nie do pojedynczego zadania.
        if vram:
            if VRAM_BASELINE_MB is not None:
                vram["vram_delta_mb"] = round(vram["vram_peak_mb"] - VRAM_BASELINE_MB, 1)
                vram["vram_baseline_mb"] = VRAM_BASELINE_MB
            fh.write(json.dumps({
                "run_id": RUN_ID, "record_type": "vram_summary",
                **cond, **vram,
            }, ensure_ascii=False) + "\n")
            fh.flush()
            print(f"      VRAM szczyt: {vram['vram_peak_mb']:.0f} MB"
                  + (f" (przyrost {vram['vram_delta_mb']:.0f} MB)"
                     if "vram_delta_mb" in vram else ""))

print(f"\nZakonczono w {(time.perf_counter() - t_start)/60:.1f} min -> {RAW_PATH}")

## ANALIZA

Ponizsza czesc czyta wylacznie plik JSONL — mozna ja uruchamiac dowolnie
wiele razy bez powtarzania benchmarku.

In [ ]:
rows = [json.loads(l) for l in RAW_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
df_all = pd.DataFrame(rows)

vram_df = df_all[df_all.get("record_type") == "vram_summary"] if "record_type" in df_all else pd.DataFrame()
if "record_type" in df_all.columns:
    record_type = df_all["record_type"]
    vram_df = df_all[record_type == "vram_summary"]
    df = df_all[record_type.isna()]
else:
    vram_df = pd.DataFrame()
    df = df_all

n_total = len(df)
df = df[df["ok"] == True].copy()  # noqa: E712
print(f"Rekordow: {n_total} | poprawnych: {len(df)} ({len(df)/max(n_total,1):.1%})")

if n_total - len(df) > 0:
    print("\nRozklad bledow:")
    print(pd.DataFrame(rows).query("ok == False")["error"].value_counts().head())

if "backend_fallback" in df.columns:
    n_fallback = int(df["backend_fallback"].fillna(False).sum())
    if n_fallback:
        print(f"\n!!! {n_fallback} odpowiedzi awaryjnych backendu (HTTP 200, brak generacji):")
        print(df[df["backend_fallback"] == True]["response_preview"].iloc[0])  # noqa: E712
        df = df[df["backend_fallback"] != True].copy()  # noqa: E712
        print(f"    -> odrzucone; pozostalo {len(df)} rekordow")
 
# Kontrola kompletnosci metryk backendu
_missing = [c for c in ["wall_total_ms", "orchestration_ms", "stage_rag_ms"]
            if c not in df.columns or df[c].isna().all()]
if _missing:
    print(f"\n!!! UWAGA: backend nie zwraca pol: {_missing}")
    print("    Dekompozycja czasu bedzie niepelna. Sprawdz instrumentacje w npc_service.py.")
 
if (df["prompt_tokens"] == 0).all():
    print("\n!!! UWAGA: prompt_tokens=0 dla WSZYSTKICH rekordow.")
    print("    Sprawdz, czy backend przekazuje prompt_eval_count w metrics.")
 
# Przeladowania wag to inne zjawisko niz normalna generacja — analizujemy je
# osobno, zamiast pozwolic im zaburzyc mediane stanu ustalonego.
reloads = df[df["model_reloaded"] == True]  # noqa: E712
print(f"\nWykrytych przeladowan modelu w fazie pomiarowej: {len(reloads)}")
if len(reloads):
    print(f"  -> mediana zimnego startu: {reloads['ttft_cold_ms'].median():.0f} ms")
 
df_warm = df[df["model_reloaded"] == False].copy()  # noqa: E712
 
# zabezpieczenie przed pustym df_warm
if df_warm.empty:
    raise RuntimeError(
        "df_warm jest pusty — wszystkie rekordy zaklasyfikowano jako przeladowanie. "
        "Sprawdz RELOAD_FLOOR_MS oraz wartosci load_ms w surowych danych."
    )
print(f"Rekordow w analizie stanu ustalonego: {len(df_warm)}")
 
# Metryka wyprowadzona: rzeczywisty czas oczekiwania w kolejce Ollamy.
# Ollama zwraca total_duration OBEJMUJACE czas oczekiwania na wolny slot
# przetwarzania, natomiast load/prompt_eval/eval_duration mierza WYLACZNIE
# aktywne fazy. Roznica to ograniczenie konfiguracyjne silnika
# (prawdopodobnie OLLAMA_NUM_PARALLEL=1), NIE narzut orkiestracji.
df_warm["ollama_queue_ms"] = (
    df_warm["ollama_total_ms"]
    - (df_warm["load_ms"] + df_warm["ttft_warm_ms"] + df_warm["decode_ms"])
).clip(lower=0)

 ## 7. Kontrola wspolzmiennej: klasa promptu

In [ ]:
cov = (df_warm.groupby("prompt_class")
       .agg(n=("wall_ms", "size"),
            ttft_med=("ttft_warm_ms", "median"),
            wall_med=("wall_ms", "median"),
            out_tok_med=("output_tokens", "median"))
       .round(1).reset_index())
display(cov)
 
# Rownomiernosc rozkladu klas w obrebie kazdego warunku
balance = (df_warm.groupby(["rag_k", "concurrency"])["prompt_class"]
           .nunique().reset_index(name="liczba_klas"))
if (balance["liczba_klas"] < len(TEST_PROMPTS)).any():
    print("!!! UWAGA: nie wszystkie klasy promptu wystapily w kazdym warunku.")
    print("    Zwieksz REPETITIONS do wielokrotnosci", len(TEST_PROMPTS))
    display(balance[balance["liczba_klas"] < len(TEST_PROMPTS)])
else:
    print(f"OK: wszystkie {len(TEST_PROMPTS)} klas promptu wystapily w kazdym warunku.")

## 8. Tabela zbiorcza

In [ ]:
def bootstrap_ci_median(x, n_boot=5000, alpha=0.05):
    """Przedzial ufnosci dla mediany metoda percentylowa.

    Rozklady latencji sa prawoskosne, wiec CI oparte na zalozeniu normalnosci
    byloby tu nieuprawnione. Bootstrap nie wymaga zalozen o ksztalcie rozkladu.
    """
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    if len(x) < 5:
        return (np.nan, np.nan)
    rng = np.random.default_rng(42)
    boots = np.median(rng.choice(x, size=(n_boot, len(x)), replace=True), axis=1)
    return float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def summarize(df_in: pd.DataFrame) -> pd.DataFrame:
    keys = ["rag_k", "concurrency"]
    out = []
    for name, g in df_in.groupby(keys, dropna=False):
        row = dict(zip(keys, name if isinstance(name, tuple) else (name,)))
        row["n"] = len(g)
        for label, col in [("ttft", "ttft_warm_ms"), ("decode", "decode_ms"),
                           ("wall", "wall_ms"), ("orch", "orchestration_ms"),
                           ("transport", "transport_ms"),
                           ("ollama_queue", "ollama_queue_ms"),
                           ("batch", "batch_wall_ms")]:
            if col not in g:
                continue

            v = g[col].dropna()
            if v.empty:
                continue
            row[f"{label}_med"] = round(v.median(), 1)
            lo, hi = bootstrap_ci_median(v)
            row[f"{label}_ci95"] = f"[{lo:.0f}, {hi:.0f}]"
            row[f"{label}_p95"] = round(np.percentile(v, 95), 1)

        row["tps_med"] = round(df_in.loc[g.index, "tps"].median(), 1)
        row["prompt_tok_med"] = round(g["prompt_tokens"].median(), 0)
        row["out_tok_med"] = round(g["output_tokens"].median(), 0)
        # Operacyjna definicja grywalnosci: odsetek odpowiedzi miesczacych sie
        # w budzecie maskowania animacja.
        row["pct_pod_2s"] = round((g["wall_ms"] < T_MASKED_MS).mean() * 100, 1)
        out.append(row)
    if not out:
        return pd.DataFrame(columns=keys)
    return pd.DataFrame(out).sort_values(keys)


summary = summarize(df_warm)
summary.to_csv(OUT / "tables" / "summary.csv", index=False)
display(summary)

In [ ]:
# Tabela VRAM — bezposrednia odpowiedz na pytanie o wykonalnosc sprzetowa.
if not vram_df.empty:
    cols = ["rag_k", "vram_peak_mb"]
    if "vram_delta_mb" in vram_df.columns:
        cols.append("vram_delta_mb")
    vram_tab = (vram_df.groupby("rag_k")[cols[1:]].max().reset_index())
    vram_tab["VRAM szczyt [GB]"] = (vram_tab["vram_peak_mb"] / 1024).round(2)
    vram_tab.to_csv(OUT / "tables" / "vram.csv", index=False)
    display(vram_tab)
    print(f"Karta: {ENV.get('gpu_name')} ({ENV.get('gpu_total_vram_mb')} MB)")
    print("UWAGA: nvidia-smi raportuje zajetosc calej karty. Kolumna")
    print("       vram_delta_mb podaje przyrost wzgledem stanu przed przebiegiem.")

## 9. Testy istotności statystycznej
  - Kruskal-Wallis: czy poziom k rozniocuje TTFT (>2 grupy)
  - Spearman:       czy zaleznosc k -> TTFT jest monotoniczna
  - Mann-Whitney U: porownania parami dla wspolbieznosci

In [ ]:
# def epsilon_squared(h_stat: float, n: int, k_groups: int) -> float:
#     """Wielkosc efektu dla testu Kruskala-Wallisa."""
#     if n - k_groups <= 0:
#         return float("nan")
#     return (h_stat - k_groups + 1) / (n - k_groups)
 
 
# def rank_biserial(u_stat: float, n1: int, n2: int) -> float:
#     """Wielkosc efektu dla testu Manna-Whitneya (korelacja rangowo-dwuseryjna)."""
#     return 1 - (2 * u_stat) / (n1 * n2)
 
 
# def interpret_eps(e: float) -> str:
#     if np.isnan(e):
#         return "nieokreslona"
#     a = abs(e)
#     return "mala" if a < 0.06 else ("umiarkowana" if a < 0.14 else "duza")
 
 
# def interpret_rb(r: float) -> str:
#     a = abs(r)
#     return "mala" if a < 0.3 else ("umiarkowana" if a < 0.5 else "duza")
 
 
# stat_rows = []

# # --- Efekt 1: wplyw k na TTFT (przy N=1, aby odizolowac od kolejkowania) ---
# d_k = df_warm[df_warm["concurrency"] == 1]
# groups_k = [g["ttft_warm_ms"].dropna().values
#             for _, g in d_k.groupby("rag_k") if len(g) >= 3]
# k_levels = sorted(d_k["rag_k"].dropna().unique())
 
# if len(groups_k) >= 2:
#     h, p = stats.kruskal(*groups_k)
#     n_all = sum(len(g) for g in groups_k)
#     eps = epsilon_squared(h, n_all, len(groups_k))
#     stat_rows.append({
#         "efekt": "k -> TTFT (N=1)", "test": "Kruskal-Wallis",
#         "statystyka": round(h, 2), "df": len(groups_k) - 1,
#         "p": f"{p:.2e}" if p < 1e-3 else round(p, 4),
#         "wielkosc_efektu": f"eps^2 = {eps:.3f}",
#         "interpretacja": interpret_eps(eps),
#         "n": n_all,
#     })
 
#     rho, p_rho = stats.spearmanr(d_k["rag_k"], d_k["ttft_warm_ms"])
#     stat_rows.append({
#         "efekt": "k -> TTFT (monotonicznosc)", "test": "Spearman",
#         "statystyka": round(rho, 3), "df": "-",
#         "p": f"{p_rho:.2e}" if p_rho < 1e-3 else round(p_rho, 4),
#         "wielkosc_efektu": f"rho = {rho:.3f}",
#         "interpretacja": interpret_rb(rho),
#         "n": len(d_k),
#     })
 
#     # Mechanizm posredniczacy: k -> dlugosc promptu
#     rho_t, p_t = stats.spearmanr(d_k["rag_k"], d_k["prompt_tokens"])
#     stat_rows.append({
#         "efekt": "k -> dlugosc promptu", "test": "Spearman",
#         "statystyka": round(rho_t, 3), "df": "-",
#         "p": f"{p_t:.2e}" if p_t < 1e-3 else round(p_t, 4),
#         "wielkosc_efektu": f"rho = {rho_t:.3f}",
#         "interpretacja": interpret_rb(rho_t),
#         "n": len(d_k),
#     })
 
# # --- Efekt 2: wplyw wspolbieznosci na czas calkowity ---
# mid_k = RAG_KS[len(RAG_KS) // 2]
# d_n = df_warm[df_warm["rag_k"] == mid_k] if mid_k is not None else df_warm
 
# groups_n = [g["wall_ms"].dropna().values
#             for _, g in d_n.groupby("concurrency") if len(g) >= 3]
# if len(groups_n) >= 2:
#     h, p = stats.kruskal(*groups_n)
#     n_all = sum(len(g) for g in groups_n)
#     eps = epsilon_squared(h, n_all, len(groups_n))
#     stat_rows.append({
#         "efekt": f"N -> czas calkowity (k={mid_k})", "test": "Kruskal-Wallis",
#         "statystyka": round(h, 2), "df": len(groups_n) - 1,
#         "p": f"{p:.2e}" if p < 1e-3 else round(p, 4),
#         "wielkosc_efektu": f"eps^2 = {eps:.3f}",
#         "interpretacja": interpret_eps(eps),
#         "n": n_all,
#     })
 
# # Porownania parami N=1 vs pozostale
# base = d_n[d_n["concurrency"] == 1]["wall_ms"].dropna()
# for c in sorted(d_n["concurrency"].unique()):
#     if c == 1:
#         continue
#     other = d_n[d_n["concurrency"] == c]["wall_ms"].dropna()
#     if len(base) < 3 or len(other) < 3:
#         continue
#     u, p = stats.mannwhitneyu(base, other, alternative="two-sided")
#     rb = rank_biserial(u, len(base), len(other))
#     stat_rows.append({
#         "efekt": f"N=1 vs N={c} (czas calkowity)", "test": "Mann-Whitney U",
#         "statystyka": round(u, 1), "df": "-",
#         "p": f"{p:.2e}" if p < 1e-3 else round(p, 4),
#         "wielkosc_efektu": f"r_rb = {rb:.3f}",
#         "interpretacja": interpret_rb(rb),
#         "n": len(base) + len(other),
#     })
 
# # --- Efekt 3 (kontrolny): czy narzut orkiestracji zalezy od N? ---
# # Oczekiwanie: BRAK istotnej zaleznosci. To potwierdza teze, ze wzrost
# # opoznienia przy rosnacym N pochodzi z kolejki silnika, nie z architektury.
# if "orchestration_ms" in d_n.columns and d_n["orchestration_ms"].notna().any():
#     groups_o = [g["orchestration_ms"].dropna().values
#                 for _, g in d_n.groupby("concurrency") if g["orchestration_ms"].notna().sum() >= 3]
#     if len(groups_o) >= 2:
#         h, p = stats.kruskal(*groups_o)
#         n_all = sum(len(g) for g in groups_o)
#         eps = epsilon_squared(h, n_all, len(groups_o))
#         stat_rows.append({
#             "efekt": "N -> narzut orkiestracji (kontrolny)", "test": "Kruskal-Wallis",
#             "statystyka": round(h, 2), "df": len(groups_o) - 1,
#             "p": f"{p:.2e}" if p < 1e-3 else round(p, 4),
#             "wielkosc_efektu": f"eps^2 = {eps:.3f}",
#             "interpretacja": interpret_eps(eps),
#             "n": n_all,
#         })
 
# stat_tab = pd.DataFrame(stat_rows)
# stat_tab.to_csv(OUT / "tables" / "testy_istotnosci.csv", index=False)
# display(stat_tab)
 
# print("\nUWAGA INTERPRETACYJNA:")
# print("Przy n rzedu setek obserwacji nawet roznica pozbawiona znaczenia")
# print("praktycznego osiaga p < 0,05. W tekscie pracy nalezy raportowac")
# print("wielkosc efektu obok wartosci p i odnosic roznice do progow")
# print("percepcyjnych, a nie wylacznie do istotnosci statystycznej.")

## 10. Wykresy

In [ ]:
def save_fig(fig, name):
    for ext in ("png", "pdf"):  # PDF = grafika wektorowa do wersji drukowanej
        fig.savefig(OUT / "figures" / f"{name}.{ext}")


# --- Rys. 1: wplyw kontekstu RAG na TTFT + mechanizm ------------------------
if len(RAG_KS) > 1:
    d1 = df_warm[df_warm["concurrency"] == 1]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    sns.lineplot(data=d1, x="rag_k", y="ttft_warm_ms", hue="model", marker="o",
                 estimator="median", errorbar=("pi", 50), ax=axes[0])
    axes[0].set(xlabel="Liczba wektorow RAG (k)", ylabel="TTFT [ms]",
                title="Time To First Token wobec kontekstu RAG")
    sns.lineplot(data=d1, x="rag_k", y="prompt_tokens", hue="model", marker="s",
                 estimator="median", ax=axes[1])
    axes[1].set(xlabel="Liczba wektorow RAG (k)", ylabel="Dlugosc promptu [tokeny]",
                title="Mechanizm: k -> rozmiar promptu -> czas prefill")
    fig.tight_layout()
    save_fig(fig, "rys1_wplyw_rag")
    plt.show()

In [ ]:
# --- Rys. 2: skalowanie przy wspolbieznosci --------------------------------
mid_k = RAG_KS[len(RAG_KS) // 2]
d2 = df_warm[df_warm["rag_k"] == mid_k] if mid_k is not None else df_warm

fig, ax = plt.subplots(figsize=(7.5, 4.5))
sns.boxplot(data=d2, x="concurrency", y="wall_ms", hue="model", showfliers=False, ax=ax)
for y, lbl, ls in [(T_FLOW_MS, "1 s — plynnosc interakcji (Nielsen)", "--"),
                   (T_MASKED_MS, "2 s — budzet maskowania (def. wlasna)", "-.")]:
    ax.axhline(y, ls=ls, lw=1.2, color="crimson")
    ax.text(ax.get_xlim()[1] * 0.99, y, f" {lbl}", va="bottom", ha="right",
            fontsize=8, color="crimson")
ax.set(xlabel="Liczba jednoczesnych agentow (N)", ylabel="Czas calkowity [ms]",
       title=f"Skalowanie latencji wobec wspolbieznosci (k={mid_k})")
fig.tight_layout()
save_fig(fig, "rys6_2_wspolbieznosc")
plt.show()

In [ ]:
# --- Rys. 3: dekompozycja budzetu opoznien ---------------------------------
# Pokazuje, ile kosztuje Twoj orkiestrator wzgledem samej inferencji.
comp_cols = ["transport_ms", "orchestration_ms", "ttft_warm_ms", "decode_ms"]
comp_cols = [c for c in comp_cols if c in df_warm.columns and df_warm[c].notna().any()]
 
d3 = (df_warm[df_warm["concurrency"] == 1]
      .groupby("rag_k", dropna=False)[comp_cols]
      .median())
 
labels = {"transport_ms": "Transport HTTP i walidacja",
          "orchestration_ms": "Orkiestracja (RAG, budowa promptu)",
          "ttft_warm_ms": "Prefill (TTFT)",
          "decode_ms": "Dekodowanie"}
d3.columns = [labels[c] for c in d3.columns]
 
fig, ax = plt.subplots(figsize=(7.5, 4.5))
d3.plot(kind="bar", stacked=True, ax=ax, width=0.7,
        color=["#d9d9d9", "#8da0cb", "#fc8d62", "#66c2a5"][-len(d3.columns):])
ax.set(xlabel="Liczba pobieranych wspomnien (k)", ylabel="Czas [ms], mediana",
       title="Dekompozycja budzetu opoznien (N=1)")
ax.legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "rys6_3_dekompozycja")
plt.show()

In [ ]:
# --- Rys. 4: kolejkowanie Ollamy + spadek przepustowosci -----------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
 
sns.boxplot(data=df_warm[df_warm["rag_k"] == mid_k], x="concurrency",
            y="ollama_queue_ms", showfliers=False, ax=axes[0], color="#c0504d")
axes[0].set(xlabel="Liczba jednoczesnych agentow (N)",
            ylabel="Czas oczekiwania w kolejce [ms]",
            title=f"Kolejkowanie generacji (k={mid_k})")
 
sns.lineplot(data=df_warm, x="concurrency", y="tps", marker="o",
             estimator="median", errorbar=("pi", 50), ax=axes[1], color="#4f81bd")
axes[1].set(xlabel="Liczba jednoczesnych agentow (N)", ylabel="Tokeny / s",
            title="Przepustowosc dekodowania na agenta")
 
fig.suptitle("Zrodlo narzutu przy wspolbieznosci: kolejka silnika inferencji")
fig.tight_layout()
save_fig(fig, "rys6_4_kolejkowanie")
plt.show()

In [ ]:
# --- Rys. 5: dystrybuanta empiryczna ---------------------------------------
# Czytelniejsza niz histogram dla ogonow rozkladu — a to ogony decyduja
# o tym, czy gracz odczuje "zacinanie sie" dialogow.
fig, ax = plt.subplots(figsize=(7.5, 4.5))
d4 = df_warm[df_warm["concurrency"].isin([1, max(CONCURRENCIES)])].copy()
d4["warunek"] = d4["model"] + ", N=" + d4["concurrency"].astype(str) # type: ignore
sns.ecdfplot(data=d4, x="wall_ms", hue="warunek", ax=ax, lw=1.6)
ax.axvline(T_MASKED_MS, ls="-.", color="crimson", lw=1.2)
ax.set(xlabel="Czas calkowity [ms]", ylabel="Frakcja odpowiedzi",
       title="Dystrybuanta empiryczna czasu odpowiedzi")
fig.tight_layout()
save_fig(fig, "rys4_dystrybuanta")
plt.show()


## 11. Podsumowanie do rozdzialu wynikow

In [ ]:
best = df_warm[df_warm["concurrency"] == 1]
print("=" * 70)
print("PODSUMOWANIE TESTU 1")
print("=" * 70)
print(f"Model:             {MODEL_NAME}")
print(f"Sprzet:            {ENV.get('gpu_name')} / {ENV.get('gpu_total_vram_mb')} MB VRAM")
print(f"Pomiarow:          {len(df_warm)} (po odrzuceniu rozgrzewki i przeladowan)")
print(f"Powtorzen/warunek: {REPETITIONS} | warunkow: {len(conditions)}")
print("-" * 70)
print("STAN USTALONY, N=1:")
print(f"   TTFT                 mediana {best['ttft_warm_ms'].median():7.0f} ms "
      f"| p95 {np.percentile(best['ttft_warm_ms'].dropna(), 95):7.0f} ms")
print(f"   Czas calkowity       mediana {best['wall_ms'].median():7.0f} ms "
      f"| p95 {np.percentile(best['wall_ms'].dropna(), 95):7.0f} ms")
if best["orchestration_ms"].notna().any():
    print(f"   Narzut orkiestracji  mediana {best['orchestration_ms'].median():7.1f} ms")
if best["transport_ms"].notna().any():
    print(f"   Transport i walidacja mediana {best['transport_ms'].median():6.1f} ms")
print(f"   Przepustowosc        mediana {best['tps'].median():7.1f} tok/s")
print(f"   Ponizej 2 s          {(best['wall_ms'] < T_MASKED_MS).mean()*100:7.1f} %")
 
print("-" * 70)
print("WSPOLBIEZNOSC:")
for c in sorted(df_warm["concurrency"].unique()):
    g = df_warm[(df_warm["concurrency"] == c) & (df_warm["rag_k"] == mid_k)]
    if g.empty:
        continue
    print(f"   N={c}: wall {g['wall_ms'].median():6.0f} ms "
          f"| kolejka {g['ollama_queue_ms'].median():6.0f} ms "
          f"| ponizej 2 s {(g['wall_ms'] < T_MASKED_MS).mean()*100:5.1f} %")
 
if len(reloads):
    print("-" * 70)
    print(f"Zimny start (oportunistyczny): mediana {reloads['ttft_cold_ms'].median():.0f} ms")
 
print("=" * 70)
print(f"\nTabele  -> {OUT/'tables'}")
print(f"Wykresy -> {OUT/'figures'}")